# POC-09A — Scenario Agent + GPT-5.6 Luna

Notebook separado para testar o conversacional do Scenario Agent.

Fluxo:

```text
Usuário
   ↓
GPT-5.6 Luna
   ↓
ScenarioAgentOpenAI
   ↓
ScenarioAgent
   ↓
ScenarioConversation
   ↓
ScenarioBuilder
   ↓
scenario.toml
```

Neste POC não usamos MCP nem Planning Service.

In [ ]:
import json
import os
import sys
from pathlib import Path

PLANAPP_REPO = Path.home() / "work" / "planapp-mcp"
WORKSPACE = Path.home() / "work" / "cisei_workspace"
PROJECT_PATH = WORKSPACE / "projects" / "scenario_agent_jupyter_poc"
PLUGIN_ROOT = PLANAPP_REPO / "cisei-planning-engineering"

PROJECT_PATH.mkdir(parents=True, exist_ok=True)

if str(PLANAPP_REPO) not in sys.path:
    sys.path.insert(0, str(PLANAPP_REPO))

if not PLUGIN_ROOT.is_dir():
    raise RuntimeError(
        f"Plugin CISEI não encontrado em: {PLUGIN_ROOT}"
    )

print("PLANAPP_REPO:", PLANAPP_REPO)
print("PROJECT_PATH:", PROJECT_PATH)
print("PLUGIN_ROOT:", PLUGIN_ROOT)


In [ ]:
from cisei_planning_sdk.project import PlanningProject
from scenario_agent import ScenarioAgent
from scenario_agent_openai import ScenarioAgentOpenAI
from scenario_domain.providers import CISEIPluginDomainContextProvider


In [ ]:
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY não está definida no ambiente do notebook."
    )

print("Modelo:", MODEL)

In [ ]:
project = PlanningProject(PROJECT_PATH)

domain_context_provider = CISEIPluginDomainContextProvider(
    plugin_path=PLUGIN_ROOT
)

agent = ScenarioAgentOpenAI(
    project,
    model=MODEL,
    domain_context_provider=domain_context_provider,
)

print(agent)
print("Plugin provider:", domain_context_provider.provider_id)
print("Plugin version:", domain_context_provider.provider_version)
print("Context version:", domain_context_provider.context_version)


In [ ]:
def conversar(mensagem: str):
    resultado = agent.chat(mensagem)

    print("\n" + "=" * 72)
    print("USUÁRIO:")
    print(mensagem)

    print("\nRESULTADO DA APLICAÇÃO:")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
    print("=" * 72)

    return resultado

## Início da conversa

A primeira mensagem deve iniciar a coleta do cenário.

In [ ]:
#resultado = conversar("Quero criar um cenário LTE.")

## Conversação interativa

Execute a célula abaixo para continuar a conversa diretamente pelo notebook.

In [ ]:
while True:
    mensagem = input("\nVocê: ").strip()

    if mensagem.lower() in {"sair", "exit", "quit"}:
        print("Sessão encerrada pelo usuário.")
        break

    if not mensagem:
        continue

    resultado = agent.chat(mensagem)

    print("\nAplicação:")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
print("MESSAGES INICIAIS:")
print(agent.messages)

In [ ]:
import json

print(json.dumps(
    agent.scenario_agent.conversation.session.builder.spec,
    ensure_ascii=False,
    indent=2
))

print("\nESTADO:")
print(json.dumps(
    agent.scenario_agent.conversation_state(),
    ensure_ascii=False,
    indent=2
))

In [ ]:
print(agent.messages)

In [ ]:
toml_text = agent.scenario_agent.conversation.session.builder.generate_toml()
print(toml_text)

In [ ]:
from pathlib import Path

toml_path = Path("scenario.toml")
toml_path.write_text(toml_text, encoding="utf-8")

print(f"Arquivo salvo em: {toml_path.resolve()}")

In [ ]:
import json

print(json.dumps(
    agent.scenario_agent.conversation.session.builder.spec,
    ensure_ascii=False,
    indent=2
))

In [ ]:
for i, msg in enumerate(agent.messages):
    print(f"\n--- MESSAGE {i} ---")
    print(msg)

## Estado atual do cenário

In [ ]:
state = agent.scenario.conversation_state()
print(json.dumps(state, ensure_ascii=False, indent=2))

## Snapshot completo

In [ ]:
snapshot = agent.scenario.snapshot()
print(json.dumps(snapshot, ensure_ascii=False, indent=2))

## Finalização

A aplicação somente gera o arquivo depois que o cenário está completo, válido e explicitamente confirmado.

In [ ]:
if (
    agent.scenario.complete
    and agent.scenario.valid
    and agent.scenario.confirmed
):
    resultado_final = agent.scenario.process_finalize()
    print(json.dumps(resultado_final, ensure_ascii=False, indent=2))
else:
    print("O cenário ainda não está pronto para finalização.")
    print("complete =", agent.scenario.complete)
    print("valid    =", agent.scenario.valid)
    print("confirmed=", agent.scenario.confirmed)

## Verificação do scenario.toml

In [ ]:
if agent.scenario_path.exists():
    print(agent.scenario_path.read_text(encoding="utf-8"))
else:
    print("scenario.toml ainda não foi gerado.")

## Critérios do POC-09A

- Conversação real com GPT-5.6 Luna.
- Nenhum parâmetro técnico inventado pelo agente.
- Campos ausentes devem ser solicitados ao usuário.
- Estado do cenário controlado pela aplicação.
- Validação executada pela aplicação.
- Resumo e confirmação explícita antes da geração.
- `scenario.toml` gerado pela aplicação.
- Nenhum MCP utilizado neste POC.
- Nenhum Planning Service utilizado neste POC.
- Contexto de domínio fornecido por `DomainContextProvider`.
- Plugin utilizado somente como conhecimento de domínio.
- Exemplos do plugin não são tratados como defaults.
- Propostas e decisões do plugin não sobrescrevem o estado da aplicação.
- Plugin não executa MCP, Planning Service ou operações de cenário.


In [ ]:
from pathlib import Path

p = Path("/home/jovyan/work/planapp-mcp/scenario_agent_openai.py")

text = p.read_text()

text = text.replace(
    "from planning_project import PlanningProject",
    "from cisei_planning_sdk.project import PlanningProject",
)

p.write_text(text)

print("Arquivo atualizado:", p)

In [ ]:
from pathlib import Path

p = Path("/home/jovyan/work/planapp-mcp/scenario_agent_openai.py")

for i, line in enumerate(p.read_text().splitlines(), 1):
    if "PlanningProject" in line:
        print(f"{i}: {line}")

In [ ]:
import sys

sys.modules.pop("scenario_agent_openai", None)

import scenario_agent_openai

print("IMPORT OK")
print(scenario_agent_openai.__file__)